# Exercise Week 6
Fine-Tuning Frontier Model (GPT-4.1-nano) with OpenAI API

In [ ]:
#Imports
import os
import json
import random
import pandas as pd
import tiktoken
from tqdm import tqdm
from datasets import load_dataset
from openai import OpenAI

In [ ]:
#Constants
BASE_MODEL       = "gpt-4.1-nano-2025-04-14"          
TRAIN_FILE       = "alpaca_train.jsonl"    
VAL_FILE         = "alpaca_val.jsonl"     
N_TRAIN_SAMPLES  = 300                    
N_VAL_SAMPLES    = 100                     
MAX_TOKENS       = 512                    
RANDOM_SEED      = 42

In [ ]:
#Environment Variables
from dotenv import load_dotenv
load_dotenv(override=True)
random.seed(RANDOM_SEED)
client = OpenAI()
print(f"Using model: {BASE_MODEL}")
print(f"Training samples: {N_TRAIN_SAMPLES} | Validation samples: {N_VAL_SAMPLES}")

In [ ]:
# Load the dataset from HuggingFace Hub
print("Loading dataset...")
dataset = load_dataset("tatsu-lab/alpaca", split="train")

df = dataset.to_pandas()
print(f"Total examples: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")
df.head(3)

In [ ]:
print("=== Sample Example ===")
sample = df.sample(1, random_state=1).iloc[0]
print(f"Instruction : {sample['instruction']}")
print(f"Input       : {sample['input']}")
print(f"Output      : {sample['output']}")

print("\n=== Output Length Distribution ===")
df["output_len"] = df["output"].str.len()
print(df["output_len"].describe().round(1))

print(f"\nExamples with additional input context: {(df['input'].str.strip() != '').sum()}")

In [ ]:
# Curate & Filter Data
def is_quality_example(row: pd.Series) -> bool:
    """Filter out low-quality or problematic examples."""
    instruction = str(row["instruction"]).strip()
    output      = str(row["output"]).strip()

    # Must have meaningful instruction and output
    if len(instruction) < 10 or len(output) < 10:
        return False

    # Skip very long outputs that bloat token count
    if len(output) > 2000:
        return False

    # Skip outputs that look like errors or placeholders
    bad_phrases = ["I cannot", "As an AI language model, I", "[INST]", "<|"]
    if any(p in output for p in bad_phrases):
        return False

    return True

mask     = df.apply(is_quality_example, axis=1)
df_clean = df[mask].reset_index(drop=True)

print(f"Before filtering : {len(df):,}")
print(f"After filtering  : {len(df_clean):,}")
print(f"Removed          : {len(df) - len(df_clean):,} examples")

In [ ]:
# Convert to OpenAI Chat Format
SYSTEM_PROMPT = (
    "You are a helpful, precise, and knowledgeable assistant. "
    "Answer questions clearly and follow instructions carefully."
)

def row_to_chat(row: pd.Series) -> dict:
    """Convert an Alpaca row to OpenAI fine-tune message format."""
    instruction = str(row["instruction"]).strip()
    inp         = str(row["input"]).strip()
    output      = str(row["output"]).strip()

    # Merge instruction + optional context input
    if inp:
        user_content = f"{instruction}\n\nContext:\n{inp}"
    else:
        user_content = instruction

    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": output},
        ]
    }

# Convert all cleaned rows
all_examples = [row_to_chat(row) for _, row in df_clean.iterrows()]
print(f"Converted {len(all_examples):,} examples to chat format.")
print("\nSample formatted example:")
print(json.dumps(all_examples[0], indent=2))

In [ ]:
# remove too big tokens from the dataset
encoding = tiktoken.get_encoding("cl100k_base")  # encoding used by GPT-4 family

def count_tokens(example: dict) -> int:
    total = 0
    for msg in example["messages"]:
        total += len(encoding.encode(msg["content"]))
    return total

print("Counting tokens (this may take a moment)...")
token_counts = [count_tokens(ex) for ex in tqdm(all_examples)]

token_series = pd.Series(token_counts)
print(f"\nToken stats across all examples:")
print(token_series.describe().round(1))

# Filter out examples that exceed the max token limit
valid_pairs   = [(ex, tc) for ex, tc in zip(all_examples, token_counts) if tc <= MAX_TOKENS]
filtered_out  = len(all_examples) - len(valid_pairs)
print(f"\nExamples within {MAX_TOKENS} token limit : {len(valid_pairs):,}")
print(f"Filtered out (too long)              : {filtered_out}")

In [ ]:
#split the dataset into train and validation sets
valid_examples = [ex for ex, _ in valid_pairs]
random.shuffle(valid_examples)

assert len(valid_examples) >= N_TRAIN_SAMPLES + N_VAL_SAMPLES, (
    f"Not enough examples! Have {len(valid_examples)}, "
    f"need {N_TRAIN_SAMPLES + N_VAL_SAMPLES}. Lower N_TRAIN_SAMPLES or N_VAL_SAMPLES."
)

train_examples = valid_examples[:N_TRAIN_SAMPLES]
val_examples   = valid_examples[N_TRAIN_SAMPLES:N_TRAIN_SAMPLES + N_VAL_SAMPLES]

def write_jsonl(examples: list, path: str):
    with open(path, "w", encoding="utf-8") as f:
        for ex in examples:
            f.write(json.dumps(ex) + "\n")
    print(f"Wrote {len(examples):,} examples → {path}")

write_jsonl(train_examples, TRAIN_FILE)
write_jsonl(val_examples,   VAL_FILE)

In [ ]:
#validate the jsonl files
def validate_jsonl(path: str):
    """Basic validation matching OpenAI's fine-tuning requirements."""
    errors = []
    with open(path, "r") as f:
        for i, line in enumerate(f):
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                errors.append(f"Line {i}: Invalid JSON — {e}")
                continue

            msgs = obj.get("messages", [])
            if not msgs:
                errors.append(f"Line {i}: Missing 'messages' key")
                continue

            roles = [m.get("role") for m in msgs]
            if "assistant" not in roles:
                errors.append(f"Line {i}: No assistant message found")

            for m in msgs:
                if not m.get("content", "").strip():
                    errors.append(f"Line {i}: Empty content in role '{m.get('role')}'")

    if errors:
        print(f" {len(errors)} errors found in {path}:")
        for e in errors[:10]:
            print(f"   {e}")
    else:
        print(f" {path} passed validation")

validate_jsonl(TRAIN_FILE)
validate_jsonl(VAL_FILE)

In [ ]:
#upload the files to openai
def upload_file(path: str) -> str:
    """Upload a JSONL file to OpenAI and return the file ID."""
    print(f"Uploading {path}...")
    with open(path, "rb") as f:
        response = client.files.create(file=f, purpose="fine-tune")
    print(f"  ↳ File ID: {response.id} | Status: {response.status}")
    return response.id

train_file_id = upload_file(TRAIN_FILE)
val_file_id   = upload_file(VAL_FILE)

In [ ]:
# Hyperparameters
HYPERPARAMS = {
    "n_epochs": 1,           
    "batch_size": 8,        
    
}

print(f"Launching fine-tune job on {BASE_MODEL}...")
job = client.fine_tuning.jobs.create(
    training_file=train_file_id,
    validation_file=val_file_id,
    model=BASE_MODEL,
    hyperparameters=HYPERPARAMS,
    suffix="alpaca-instruct",   #append to the fine-tuned model name
)

JOB_ID = job.id
print(f"\n Fine-tune job created!")
print(f"   Job ID : {JOB_ID}")
print(f"   Status : {job.status}")
print(f"   Model  : {job.model}")

Launching fine-tune job on gpt-4.1-nano-2025-04-14...

 Fine-tune job created!
   Job ID : ftjob-HqCJIQsi5UZeQmzJeGCQU0lN
   Status : validating_files
   Model  : gpt-4.1-nano-2025-04-14


In [ ]:
# Manually check the status:
job_status = client.fine_tuning.jobs.retrieve(JOB_ID)
print(f"Current status : {job_status.status}")
print(f"Fine-tuned model (if done): {job_status.fine_tuned_model}")

Current status : succeeded
Fine-tuned model (if done): ft:gpt-4.1-nano-2025-04-14:rithwik-personal:alpaca-instruct:DGL5LDCD


https://platform.openai.com/finetune

In [ ]:
#plot the training metrics
import matplotlib.pyplot as plt

def plot_training_metrics(job_id: str):
    """Extract and plot loss curves from fine-tuning events."""
    events = client.fine_tuning.jobs.list_events(job_id, limit=100)
    steps, train_loss, val_loss = [], [], []

    for event in events.data:
        data = getattr(event, "data", {})
        if isinstance(data, dict) and "train_loss" in data:
            steps.append(data.get("step", len(steps)))
            train_loss.append(data["train_loss"])
            if "valid_loss" in data:
                val_loss.append(data["valid_loss"])

    if not steps:
        print("No metric data yet. Run this after training completes.")
        return

    plt.figure(figsize=(10, 4))
    plt.plot(steps, train_loss, label="Train Loss", color="steelblue")
    if val_loss:
        plt.plot(steps[:len(val_loss)], val_loss, label="Val Loss", color="tomato")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("Fine-Tuning Loss Curve")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_training_metrics(JOB_ID)

In [ ]:
# print the fine-tuned model
FINE_TUNED_MODEL = client.fine_tuning.jobs.retrieve(JOB_ID).fine_tuned_model
print(FINE_TUNED_MODEL)

In [ ]:
# test the fine-tuned model
def ask(prompt: str, model: str = FINE_TUNED_MODEL, system: str = SYSTEM_PROMPT) -> str:
    if not model:
        return "Model not ready yet — check JOB_ID status."
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system",  "content": system},
            {"role": "user",    "content": prompt},
        ],
        temperature=0.7,
        max_tokens=512,
    )
    return response.choices[0].message.content

# --- Run test prompts ---
test_prompts = [
    "Explain the difference between supervised and unsupervised learning in simple terms.",
    "What are three practical tips for improving focus while working from home?",
]

for prompt in test_prompts:
    print(f"USER: {prompt}")
    print(f"MODEL: {ask(prompt)}")
    print("-" * 70)

In [ ]:
#compare the base and fine-tuned model
test_prompt = "Give me a step-by-step plan to learn a new language in 6 months."

print("=" * 70)
print(f"PROMPT: {test_prompt}")
print("=" * 70)

print(f"\n[BASE: {BASE_MODEL}]")
base_response = ask(test_prompt, model=BASE_MODEL)
print(base_response)

print(f"\n[FINE-TUNED: {FINE_TUNED_MODEL}]")
ft_response = ask(test_prompt, model=FINE_TUNED_MODEL)
print(ft_response)